## Getting Data from APIs (Subway, LRT, StreetCar, Bus)

In [47]:
import requests
import pandas as pd
import re
import os
import zipfile
import json
from io import BytesIO, StringIO
from datetime import datetime
from collections import defaultdict
from rapidfuzz import fuzz, process

# ----------------------------------------------------------------------
# Comprehensive incident categorization
# ----------------------------------------------------------------------
def _build_code_mapping():
    """Build dictionary mapping raw incident codes to standard categories."""
    mapping = {}

    # ---- Subway codes (EU, MU, PU, SU, TU) ----
    # Equipment / Mechanical (EU)
    eu_mech = [
        'EUAC', 'EUAL', 'EUATC', 'EUBK', 'EUBO', 'EUCA', 'EUCH', 'EUCO',
        'EUDO', 'EUECD', 'EUHV', 'EULT', 'EULV', 'EUNEA', 'EUNT', 'EUO',
        'EUPI', 'EUSC', 'EUTL', 'EUTM', 'EUTR', 'EUTRD', 'EUVA', 'EUVE', 'EUYRD'
    ]
    for code in eu_mech:
        mapping[code] = 'Equipment / Mechanical'

    mapping['EUCD'] = 'General Delay / Other'          # Consequential Delay
    mapping['EUME'] = 'Operations / Human Error'       # Maintenance Error
    mapping['EUOE'] = 'Operations / Human Error'       # Rail Cars & Shops Opr. Error
    mapping['EUOPO'] = 'Infrastructure / Track / Signals'

    # Miscellaneous (MU)
    mapping['MUD']   = 'Passenger / Security'
    mapping['MUDD']  = 'External / Environment'
    mapping['MUEC']  = 'Infrastructure / Track / Signals'
    mapping['MUESA'] = 'Operations / Human Error'
    mapping['MUFM']  = 'External / Environment'
    mapping['MUFS']  = 'External / Environment'
    mapping['MUGD']  = 'General Delay / Other'
    mapping['MUI']   = 'Passenger / Security'
    mapping['MUIE']  = 'Passenger / Security'
    mapping['MUIR']  = 'Passenger / Security'
    mapping['MUIRS'] = 'Passenger / Security'
    mapping['MUIS']  = 'Passenger / Security'
    mapping['MULD']  = 'Management / Administrative'
    mapping['MUNOA'] = 'Operations / Human Error'
    mapping['MUO']   = 'General Delay / Other'
    mapping['MUODC'] = 'Infrastructure / Track / Signals'
    mapping['MUPAA'] = 'Passenger / Security'
    mapping['MUPLA'] = 'External / Environment'
    mapping['MUPLB'] = 'External / Environment'
    mapping['MUPLC'] = 'External / Environment'
    mapping['MUPR1'] = 'Passenger / Security'
    mapping['MUSAN'] = 'Cleaning / Unsanitary'
    mapping['MUSC']  = 'Equipment / Mechanical'
    mapping['MUTD']  = 'Management / Administrative'
    mapping['MUTO']  = 'General Delay / Other'
    mapping['MUWEA'] = 'External / Environment'
    mapping['MUWR']  = 'Management / Administrative'

    # Infrastructure (PU)
    pu_infra = [
        'PUATC', 'PUCBI', 'PUCSC', 'PUCSS', 'PUDCS', 'PUMEL', 'PUMO',
        'PUOPO', 'PUSAC', 'PUSBE', 'PUSCA', 'PUSCR', 'PUSEA', 'PUSI',
        'PUSIO', 'PUSIS', 'PUSLC', 'PUSO', 'PUSRA', 'PUSSW', 'PUSTC',
        'PUSTP', 'PUSTS', 'PUSWZ', 'PUSZC', 'PUTCD', 'PUTD', 'PUTIJ',
        'PUTNT', 'PUTO', 'PUTOE', 'PUTR', 'PUTS', 'PUTSC', 'PUTSM',
        'PUTTC', 'PUTTP', 'PUTWZ'
    ]
    for code in pu_infra:
        mapping[code] = 'Infrastructure / Track / Signals'

    mapping['PUMST'] = 'Passenger / Security'
    mapping['PUTDN'] = 'External / Environment'
    mapping['PUTIS'] = 'External / Environment'
    mapping['PUSNT'] = 'General Delay / Other'

    # Security (SU)
    su_sec = [
        'SUAE', 'SUAP', 'SUBT', 'SUCOL', 'SUDP', 'SUEAS', 'SUG',
        'SUO', 'SUPOL', 'SUROB', 'SUSA', 'SUSP', 'SUUT'
    ]
    for code in su_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TU)
    mapping['TUATC'] = 'Operations / Human Error'
    mapping['TUCC']  = 'Operations / Human Error'
    mapping['TUDOE'] = 'Operations / Human Error'
    mapping['TUKEY'] = 'Operations / Human Error'
    mapping['TUML']  = 'Scheduling / Late Starts'
    mapping['TUMVS'] = 'Operations / Human Error'
    mapping['TUNIP'] = 'Operations / Human Error'
    mapping['TUNOA'] = 'Operations / Human Error'
    mapping['TUO']   = 'General Delay / Other'
    mapping['TUOPO'] = 'Operations / Human Error'
    mapping['TUOS']  = 'Operations / Human Error'
    mapping['TUS']   = 'Scheduling / Late Starts'
    mapping['TUSC']  = 'Operations / Human Error'
    mapping['TUSET'] = 'Operations / Human Error'
    mapping['TUST']  = 'External / Environment'
    mapping['TUSUP'] = 'Operations / Human Error'

    # ---- Streetcar codes (ER, MR, PR, SR, TR) ----
    # Equipment / Mechanical (ER)
    er_mech = [
        'ERAC', 'ERBO', 'ERCO', 'ERDB', 'ERDO', 'ERHV', 'ERLT', 'ERLV',
        'ERNEA', 'ERNT', 'ERO', 'ERPR', 'ERRA', 'ERTB', 'ERTC', 'ERTL',
        'ERTR', 'ERVE', 'ERWA', 'ERWS'
    ]
    for code in er_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['ERCD'] = 'General Delay / Other'
    mapping['ERME'] = 'Operations / Human Error'

    # Miscellaneous (MR)
    mapping['MRCL']  = 'Management / Administrative'
    mapping['MRD']   = 'Passenger / Security'
    mapping['MRDD']  = 'External / Environment'
    mapping['MREC']  = 'Infrastructure / Track / Signals'
    mapping['MRESA'] = 'Operations / Human Error'
    mapping['MRFS']  = 'External / Environment'
    mapping['MRIE']  = 'Passenger / Security'
    mapping['MRLD']  = 'Management / Administrative'
    mapping['MRNOA'] = 'Operations / Human Error'
    mapping['MRO']   = 'General Delay / Other'
    mapping['MRPAA'] = 'Passenger / Security'
    mapping['MRPLA'] = 'External / Environment'
    mapping['MRPLB'] = 'External / Environment'
    mapping['MRPLC'] = 'External / Environment'
    mapping['MRPR1'] = 'Passenger / Security'
    mapping['MRSAN'] = 'Cleaning / Unsanitary'
    mapping['MRSTM'] = 'Infrastructure / Track / Signals'
    mapping['MRTO']  = 'General Delay / Other'
    mapping['MRUI']  = 'Passenger / Security'
    mapping['MRUIR'] = 'Passenger / Security'
    mapping['MRWEA'] = 'External / Environment'

    # Infrastructure (PR)
    mapping['PREL']  = 'Infrastructure / Track / Signals'
    mapping['PRO']   = 'General Delay / Other'
    mapping['PRS']   = 'Infrastructure / Track / Signals'
    mapping['PRSA']  = 'Infrastructure / Track / Signals'
    mapping['PRSL']  = 'Infrastructure / Track / Signals'
    mapping['PRSO']  = 'Infrastructure / Track / Signals'
    mapping['PRSP']  = 'Infrastructure / Track / Signals'
    mapping['PRST']  = 'Passenger / Security'
    mapping['PRSW']  = 'Infrastructure / Track / Signals'
    mapping['PRTST'] = 'Infrastructure / Track / Signals'
    mapping['PRW']   = 'Infrastructure / Track / Signals'

    # Security (SR)
    sr_sec = [
        'SRAE', 'SRAP', 'SRBT', 'SRCOL', 'SRDP', 'SREAS', 'SRO',
        'SRSA', 'SRSP', 'SRUT'
    ]
    for code in sr_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TR)
    mapping['TRDOE'] = 'Operations / Human Error'
    mapping['TRNIP'] = 'Operations / Human Error'
    mapping['TRNOA'] = 'Operations / Human Error'
    mapping['TRO']   = 'General Delay / Other'
    mapping['TRSET'] = 'Operations / Human Error'
    mapping['TRST']  = 'External / Environment'
    mapping['TRTC']  = 'Operations / Human Error'

    # ---- LRT codes (EX, MX, PX, SX, TX) ----
    # Equipment / Mechanical (EX)
    ex_mech = [
        'EXAC', 'EXBK', 'EXBO', 'EXCB', 'EXCE', 'EXCO', 'EXDB', 'EXDO',
        'EXECD', 'EXGA', 'EXGF', 'EXHV', 'EXLT', 'EXNEA', 'EXNT', 'EXO',
        'EXOSC', 'EXSA', 'EXSE', 'EXTB', 'EXTM', 'EXTR', 'EXVC', 'EXVE',
        'EXWA', 'EXWM', 'EXWS', 'EXYRD'
    ]
    for code in ex_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['EXADD'] = 'Equipment / Mechanical'
    mapping['EXOE']  = 'Operations / Human Error'
    mapping['EXPD']  = 'Collision / Roadblock'
    mapping['EXPI']  = 'Collision / Roadblock'

    # Miscellaneous (MX)
    mapping['MXAFR'] = 'Infrastructure / Track / Signals'
    mapping['MXCL']  = 'Management / Administrative'
    mapping['MXCSA'] = 'Operations / Human Error'
    mapping['MXD']   = 'Passenger / Security'
    mapping['MXDD']  = 'External / Environment'
    mapping['MXESA'] = 'Operations / Human Error'
    mapping['MXFM']  = 'External / Environment'
    mapping['MXFS']  = 'External / Environment'
    mapping['MXGD']  = 'General Delay / Other'
    mapping['MXI']   = 'Passenger / Security'
    mapping['MXIC']  = 'Passenger / Security'
    mapping['MXIE']  = 'Passenger / Security'
    mapping['MXIR']  = 'Passenger / Security'
    mapping['MXIRS'] = 'Passenger / Security'
    mapping['MXIS']  = 'Passenger / Security'
    mapping['MXLDC'] = 'Management / Administrative'
    mapping['MXLDT'] = 'Management / Administrative'
    mapping['MXNCA'] = 'Operations / Human Error'
    mapping['MXNOA'] = 'Operations / Human Error'
    mapping['MXO']   = 'General Delay / Other'
    mapping['MXPAA'] = 'Passenger / Security'
    mapping['MXPF']  = 'Infrastructure / Track / Signals'
    mapping['MXPLA'] = 'External / Environment'
    mapping['MXPLB'] = 'External / Environment'
    mapping['MXPLC'] = 'External / Environment'
    mapping['MXPR']  = 'External / Environment'
    mapping['MXPR1'] = 'Passenger / Security'
    mapping['MXPU']  = 'Operations / Human Error'
    mapping['MXSAN'] = 'Cleaning / Unsanitary'
    mapping['MXTD']  = 'Management / Administrative'
    mapping['MXTO']  = 'General Delay / Other'
    mapping['MXUS']  = 'Scheduling / Late Starts'
    mapping['MXWEA'] = 'External / Environment'
    mapping['MXWR']  = 'Management / Administrative'

    # Infrastructure (PX)
    mapping['PXATC'] = 'Infrastructure / Track / Signals'
    mapping['PXDCS'] = 'Infrastructure / Track / Signals'
    mapping['PXEAS'] = 'Infrastructure / Track / Signals'
    mapping['PXEME'] = 'Operations / Human Error'
    mapping['PXEO']  = 'General Delay / Other'
    mapping['PXMEL'] = 'Infrastructure / Track / Signals'
    mapping['PXMO']  = 'General Delay / Other'
    mapping['PXMST'] = 'Passenger / Security'
    mapping['PXOV']  = 'Infrastructure / Track / Signals'
    mapping['PXSAC'] = 'Infrastructure / Track / Signals'
    mapping['PXSBE'] = 'Infrastructure / Track / Signals'
    mapping['PXSCA'] = 'Infrastructure / Track / Signals'
    mapping['PXSCR'] = 'Infrastructure / Track / Signals'
    mapping['PXSI']  = 'Infrastructure / Track / Signals'
    mapping['PXSIS'] = 'Infrastructure / Track / Signals'
    mapping['PXSNT'] = 'General Delay / Other'
    mapping['PXSO']  = 'General Delay / Other'
    mapping['PXSRA'] = 'Infrastructure / Track / Signals'
    mapping['PXSTP'] = 'Infrastructure / Track / Signals'
    mapping['PXSW']  = 'Infrastructure / Track / Signals'
    mapping['PXTD']  = 'Infrastructure / Track / Signals'
    mapping['PXTDN'] = 'External / Environment'
    mapping['PXTIS'] = 'External / Environment'
    mapping['PXTR']  = 'Infrastructure / Track / Signals'
    mapping['PXTS']  = 'Infrastructure / Track / Signals'
    mapping['PXW']   = 'Infrastructure / Track / Signals'
    mapping['PXWZ']  = 'Infrastructure / Track / Signals'

    # Security (SX)
    sx_sec = [
        'SXAE', 'SXAM', 'SXAP', 'SXAX', 'SXBT', 'SXCOL', 'SXDP',
        'SXEAS', 'SXG', 'SXO', 'SXPOL', 'SXROB', 'SXSA', 'SXSP', 'SXUEG'
    ]
    for code in sx_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TX)
    mapping['TXATC'] = 'Operations / Human Error'
    mapping['TXCC']  = 'Operations / Human Error'
    mapping['TXDOE'] = 'Operations / Human Error'
    mapping['TXLF']  = 'Scheduling / Late Starts'
    mapping['TXML']  = 'Scheduling / Late Starts'
    mapping['TXMVS'] = 'Operations / Human Error'
    mapping['TXNCA'] = 'Operations / Human Error'
    mapping['TXNIP'] = 'Operations / Human Error'
    mapping['TXNOA'] = 'Operations / Human Error'
    mapping['TXO']   = 'General Delay / Other'
    mapping['TXOI']  = 'Passenger / Security'
    mapping['TXOS']  = 'Operations / Human Error'
    mapping['TXOVS'] = 'Operations / Human Error'
    mapping['TXPD']  = 'Collision / Roadblock'
    mapping['TXPI']  = 'Collision / Roadblock'
    mapping['TXS']   = 'Scheduling / Late Starts'
    mapping['TXST']  = 'External / Environment'
    mapping['TXSUP'] = 'Operations / Human Error'
    mapping['TXSV']  = 'Operations / Human Error'

    return mapping

# Global mapping dictionary (built once)
_INCIDENT_CODE_MAP = _build_code_mapping()

def categorize_incident(code):
    """
    Convert an incident code or description into a standard category.
    Returns one of ten categories:
        - Equipment / Mechanical
        - Operations / Human Error
        - Infrastructure / Track / Signals
        - Passenger / Security
        - External / Environment
        - Scheduling / Late Starts
        - Collision / Roadblock
        - Cleaning / Unsanitary
        - Management / Administrative
        - General Delay / Other
    """
    if pd.isna(code):
        return 'Unknown'
    s = str(code).strip()
    if not s:
        return 'Unknown'

    # Step 1: If it looks like a pure code (all caps, 2-6 letters), try the code map
    if re.match(r'^[A-Z]{2,6}$', s):
        return _INCIDENT_CODE_MAP.get(s, 'General Delay / Other')

    # Step 2: Otherwise, treat as descriptive text and apply patterns
    s_lower = s.lower()

    # Priority order: more specific patterns first
    patterns = [
        (r'late (leaving|entering)|unable to maintain schedule|mainline storage', 'Scheduling / Late Starts'),
        (r'collision|road ?block', 'Collision / Roadblock'),
        (r'clean|unsanitary|disinfection', 'Cleaning / Unsanitary'),
        (r'management|clerk|training|labour dispute|work refusal', 'Management / Administrative'),
        (r'weather|ice|snow|fire|debris|force majeure|storm', 'External / Environment'),
        (r'mechanical|equipment|brakes|door.*faulty|hvac|propulsion', 'Equipment / Mechanical'),
        (r'operations?.*operator|signal violation|overshot|overspeed|not in position|supervisory', 'Operations / Human Error'),
        (r'passenger|security|assault|disorderly|bomb|alarm|unauthorized|injur', 'Passenger / Security'),
        (r'infrastructure|track|signal|power|escalator|elevator|switch|rail|debris.*controllable', 'Infrastructure / Track / Signals'),
        (r'general delay|consequential|no trouble|other', 'General Delay / Other'),
    ]

    for pattern, category in patterns:
        if re.search(pattern, s_lower):
            return category

    # Fallback
    return 'General Delay / Other'


def clean_and_standardize(df):
    """Standardize column names and enforce a uniform schema."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    column_mappings = {
        'Date': ['Report Date', 'Date', 'Incident Date', 'Date & Time'],
        'Time': ['Time', 'Incident Time'],
        'Day': ['Day'],
        'Location': ['Location', 'Station', 'Station Name', 'Stop', 'Stop Name'],
        'Incident': ['Incident', 'Code', 'Description'],
        'Min Delay': ['Min Delay', 'Delay', 'Delay Minutes', 'Delay_Minutes'],
        'Min Gap': ['Min Gap', 'Gap', 'Gap Minutes', 'Gap_Minutes'],
        'Route': ['Route', 'Route Number', 'Route No', 'Route_ID'],
        'Line': ['Line'],
        'Direction': ['Direction', 'Bound'],
        'Vehicle': ['Vehicle', 'Vehicle Number', 'Vehicle_No']
    }

    reverse_mapping = {}
    for std_name, possible in column_mappings.items():
        for name in possible:
            reverse_mapping[name] = std_name

    rename_dict = {col: reverse_mapping[col] for col in df.columns if col in reverse_mapping}
    df = df.rename(columns=rename_dict)

    # Handle Line column -> Route, Route Name
    if 'Line' in df.columns and 'Route' not in df.columns:
        def extract_route_info(line_val):
            if pd.isna(line_val):
                return pd.Series([None, None])
            line_str = str(line_val).strip()
            match = re.match(r'^(\d+)(?:\s+(.+))?$', line_str)
            if match:
                return pd.Series([match.group(1), match.group(2) if match.group(2) else ''])
            match_digits = re.match(r'^(\d+)$', line_str)
            if match_digits:
                return pd.Series([match_digits.group(1), ''])
            return pd.Series([line_str, ''])
        df[['Route', 'Route Name']] = df['Line'].apply(extract_route_info)

    if 'Route' in df.columns and 'Route Name' not in df.columns:
        df['Route Name'] = ''

    required_columns = [
        'Date', 'Route', 'Route Name', 'Time', 'Day', 'Location',
        'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle',
        'Incident_Code'                    # keep the original code
    ]
    for col in required_columns:
        if col not in df.columns:
            df[col] = None

    df = df[required_columns]
    return df


# ----------------------------------------------------------------------
# Download and merge delay data for a single mode
# ----------------------------------------------------------------------
def load_mode_delay_data(mode_name, package_id, extra_csv=None, force_csv_only=False, skip_year_filter=False):
    """Downloads all resources for a given transit mode, cleans and returns a DataFrame.

    If skip_year_filter=True, no year extraction or filtering is performed – all resources
    with allowed formats are downloaded regardless of filename year.
    """
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    current_year = datetime.now().year
    print(f"\n🚋 Processing {mode_name} data (package: {package_id})")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception(f"CKAN API request failed for {mode_name}")

    resources = data["result"]["resources"]
    print(f"📦 Found {len(resources)} resource(s)")

    year_pattern = re.compile(r"(19|20)\d{2}")
    allowed_formats = {'csv'} if force_csv_only else {'csv', 'xlsx', 'xls'}

    mode_dfs = []

    for res in resources:
        res_name = res.get("name", "Unnamed")
        datastore_active = res.get("datastore_active", False)

        # --- Year extraction and filtering (skip if skip_year_filter=True) ---
        if not skip_year_filter:
            year_match = year_pattern.search(res_name)
            if not year_match:
                print(f"  ⏭️  {res_name}: no year found, skipping")
                continue
            year = int(year_match.group(0))
            if year < 2014:
                print(f"  ⏭️  {res_name}: year {year} < 2014, skipping")
                continue
        else:
            year = None   # No year information

        # Determine format
        if datastore_active:
            fmt = 'csv'
        else:
            fmt = res.get("format", "").lower()
            if not fmt:
                url = res.get("url", "")
                if url.endswith('.csv'):
                    fmt = 'csv'
                elif url.endswith(('.xlsx', '.xls')):
                    fmt = 'xlsx'
                else:
                    fmt = 'unknown'

        if fmt not in allowed_formats:
            print(f"  ⏭️  {res_name}: format '{fmt}' not in {allowed_formats}, skipping")
            continue

        # Apply year‑based format filter only if not skipped
        if not skip_year_filter and not force_csv_only:
            if year >= current_year-1 and fmt != 'csv':
                print(f"  ⏭️  {res_name}: skipping XLSX for latest year, only CSV kept")
                continue
            elif year < current_year-1 and fmt not in ('xlsx', 'xls'):
                print(f"  ⏭️  {res_name}: skipping CSV for year < {current_year}, only XLSX kept")
                continue

        # Print info, including year if available
        if year:
            print(f"  📄 Resource: {res_name} (year={year}, format={fmt}, datastore_active={datastore_active})")
        else:
            print(f"  📄 Resource: {res_name} (format={fmt}, datastore_active={datastore_active})")

        try:
            if datastore_active:
                dump_url = f"{base_url}/datastore/dump/{res['id']}"
                dump_resp = requests.get(dump_url)
                dump_resp.raise_for_status()
                df = pd.read_csv(StringIO(dump_resp.text))
            else:
                file_url = res["url"]
                file_resp = requests.get(file_url)
                file_resp.raise_for_status()

                if fmt in ('xlsx', 'xls'):
                    excel_data = pd.read_excel(BytesIO(file_resp.content), sheet_name=None)
                    sheet_dfs = []
                    for sheet_name, sheet_df in excel_data.items():
                        if not sheet_df.empty:
                            # Keep original incident code if present
                            if 'Incident' in sheet_df.columns:
                                sheet_df['Incident_Code'] = sheet_df['Incident']
                            sheet_df = clean_and_standardize(sheet_df)
                            sheet_dfs.append(sheet_df)
                    if sheet_dfs:
                        df = pd.concat(sheet_dfs, ignore_index=True)
                    else:
                        print(f"    ⚠️ No data in any sheet, skipping")
                        continue
                else:
                    df = pd.read_csv(StringIO(file_resp.text))
                    if 'Incident' in df.columns:
                        df['Incident_Code'] = df['Incident']

            # Clean and standardize (the cleaned df now includes Incident_Code)
            df = clean_and_standardize(df)

            # Apply incident categorization directly to the raw code/description
            df['Incident'] = df['Incident_Code'].apply(categorize_incident)
            # Also set Incident_Category to the same value (used in summaries)
            df['Incident_Category'] = df['Incident']
            df['Transit'] = mode_name

            mode_dfs.append(df)
            print(f"    ✅ Loaded {len(df)} records")

        except Exception as e:
            print(f"    ❌ Error: {e}")
            continue

    if extra_csv and os.path.isfile(extra_csv):
        print(f"\n  📄 Extra local file: {extra_csv}")
        try:
            df_extra = pd.read_csv(extra_csv)
            if 'Incident' in df_extra.columns:
                df_extra['Incident_Code'] = df_extra['Incident']
            df_extra = clean_and_standardize(df_extra)
            df_extra['Incident'] = df_extra['Incident_Code'].apply(categorize_incident)
            df_extra['Incident_Category'] = df_extra['Incident']
            df_extra['Transit'] = mode_name
            mode_dfs.append(df_extra)
            print(f"    ✅ Loaded {len(df_extra)} records from extra file")
        except Exception as e:
            print(f"    ❌ Error reading extra file {extra_csv}: {e}")
    elif extra_csv:
        print(f"\n  ⏭️ Extra file {extra_csv} not found, skipping")

    if not mode_dfs:
        print(f"⚠️ No valid data loaded for {mode_name}")
        return pd.DataFrame()

    mode_combined = pd.concat(mode_dfs, ignore_index=True)
    print(f"✅ {mode_name} total records: {len(mode_combined)}")
    return mode_combined


# ----------------------------------------------------------------------
# Download GTFS static data (routes, trips, stops)
# ----------------------------------------------------------------------
def download_gtfs_schedule():
    """Download and extract routes.txt, trips.txt, stops.txt from the merged GTFS package."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    print("\n🗺️ Downloading GTFS schedule data...")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource (complete GTFS)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract required files into memory
    gtfs_data = {}
    required_files = ['routes.txt', 'trips.txt', 'stops.txt']
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")

    # Convert to DataFrames
    routes = pd.read_csv(StringIO(gtfs_data.get('routes.txt', '')))
    trips = pd.read_csv(StringIO(gtfs_data.get('trips.txt', '')))
    stops = pd.read_csv(StringIO(gtfs_data.get('stops.txt', '')))

    return routes, trips, stops


# ----------------------------------------------------------------------
# Prepare stop matching data structures
# ----------------------------------------------------------------------
def prepare_stop_matcher(stops_df):
    """Build normalized stop names and inverted index for fast matching."""
    def normalize_stop(name):
        if pd.isna(name):
            return ''
        s = name.lower()
        s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    stops = stops_df.copy()
    stops['norm_stop'] = stops['stop_name'].apply(normalize_stop)
    stops['stop_words'] = stops['norm_stop'].apply(lambda x: set(x.split()))

    stop_names = stops['stop_name'].tolist()
    stop_norms = stops['norm_stop'].tolist()
    stop_words_list = stops['stop_words'].tolist()
    stop_lats = stops['stop_lat'].tolist()
    stop_lons = stops['stop_lon'].tolist()

    word_to_indices = defaultdict(set)
    for idx, words in enumerate(stop_words_list):
        for word in words:
            word_to_indices[word].add(idx)

    return (stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices)


# ----------------------------------------------------------------------
# Clean a delay location string
# ----------------------------------------------------------------------
def clean_location(loc):
    if pd.isna(loc):
        return ''
    s = loc.lower()
    s = re.sub(r'\(.*?\)', '', s)       # remove parenthetical notes
    s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
    s = re.sub(r'\s+', ' ', s).strip()
    # Expand common direction abbreviations
    s = re.sub(r'\bw\b', ' west', s)
    s = re.sub(r'\be\b', ' east', s)
    s = re.sub(r'\bn\b', ' north', s)
    s = re.sub(r'\bs\b', ' south', s)
    return s


# ----------------------------------------------------------------------
# Match a delay location to a stop (using pre‑built structures)
# ----------------------------------------------------------------------
def match_location(loc, stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices):
    if pd.isna(loc):
        return None, None, None

    cleaned = clean_location(loc)
    words = set(cleaned.split())

    # Find candidate stops that contain *any* of the words
    candidates = set()
    for w in words:
        candidates.update(word_to_indices.get(w, set()))
    if not candidates:
        return None, None, None

    # 1. Exact subset match (all words appear)
    for idx in candidates:
        if words.issubset(stop_words_list[idx]):
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 2. Substring match (cleaned location appears in normalized stop name)
    for idx in candidates:
        if cleaned in stop_norms[idx]:
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 3. Fuzzy match using rapidfuzz
    cand_indices = list(candidates)
    cand_norms = [stop_norms[i] for i in cand_indices]
    result = process.extractOne(
        cleaned,
        cand_norms,
        scorer=fuzz.token_set_ratio,
        score_cutoff=60,
        processor=None
    )
    if result:
        best_pos = cand_norms.index(result[0])
        best_idx = cand_indices[best_pos]
        return stop_names[best_idx], stop_lats[best_idx], stop_lons[best_idx]

    return None, None, None



def generate_route_geometries():
    """
    Downloads trips.txt and shapes.txt from the merged GTFS package,
    and generates a JSON file mapping route+short_name combinations
    to their shape geometries (list of [lat, lon] points).
    The file is saved as assets/data/route_geometries.json.
    """
    import json
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "route_geometries.json")

    print("\n🗺️ Generating route geometries from GTFS...")

    # 1. Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # 2. Find the ZIP resource (non-datastore, format=ZIP)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # 3. Extract trips.txt and shapes.txt into memory
    required_files = ['trips.txt', 'shapes.txt']
    gtfs_data = {}
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")
                return

    # 4. Read trips.txt
    trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))
    # Keep only necessary columns: route_id, trip_short_name, shape_id
    trips = trips[['route_id', 'trip_short_name', 'shape_id']].drop_duplicates()
    # Convert to string and fill missing short names
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()

    # 5. Read shapes.txt
    shapes = pd.read_csv(StringIO(gtfs_data['shapes.txt']))
    shapes = shapes[['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']]
    shapes = shapes.sort_values(['shape_id', 'shape_pt_sequence'])

    # 6. Build geometry per shape_id
    shape_geometries = {}
    for shape_id, group in shapes.groupby('shape_id'):
        # Create list of [lat, lon] pairs in order
        coords = group[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
        shape_geometries[shape_id] = coords

    # 7. Merge trips with shape geometries
    # For each (route_id, trip_short_name) we need a geometry.
    # There might be multiple trips with same route+short_name but different shape_id.
    # We'll take the first shape_id for each combination (assuming consistency).
    # Alternatively, we could keep all, but the user likely wants one geometry per route variant.
    route_geometries = {}
    # Group trips by (route_id, trip_short_name) and take first shape_id
    for (route_id, short_name), group in trips.groupby(['route_id', 'trip_short_name']):
        shape_id = group.iloc[0]['shape_id']  # first shape_id
        if shape_id in shape_geometries:
            key = route_id + short_name  # e.g., "100A"
            route_geometries[key] = shape_geometries[shape_id]
        else:
            print(f"⚠️ Shape ID {shape_id} not found for route {key}")

    # 8. Save to JSON
    with open(output_file, 'w') as f:
        json.dump(route_geometries, f, indent=2)
    print(f"✅ Route geometries saved to {output_file}")
    return route_geometries



# ----------------------------------------------------------------------
# Build route variants map from trips.txt
# ----------------------------------------------------------------------
def get_route_variants(trips_df):
    """Return dict {route_id: list_of_full_route_names} e.g. {'129': ['129A','129B','129C']}"""
    trips = trips_df.copy()
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()
    # Keep only rows with a non‑empty short name
    trips = trips[trips['trip_short_name'] != '']
    variants = trips.groupby('route_id')['trip_short_name'].apply(list).to_dict()
    route_variants = {}
    for route, short_names in variants.items():
        route_variants[route] = sorted([f"{route}{sn}" for sn in short_names if sn])
    return route_variants


# ----------------------------------------------------------------------
# Main: load all modes, merge, enrich, and produce summaries
# ----------------------------------------------------------------------
def load_all_ttc_delay_data():
    """Complete pipeline: download all delay data, GTFS, enrich, and output summaries."""

    # Define output directory
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output files will be saved to: {output_dir}")

    # 1. Download delay data for all modes
    modes = [
        ("Bus", "ttc-bus-delay-data"),
        ("Streetcar", "ttc-streetcar-delay-data"),
        ("Subway", "ttc-subway-delay-data"),
        ("LRT", "ttc-lrt-delay-data")
    ]

    all_dfs = []
    for mode_name, pkg_id in modes:
        if mode_name == "LRT":
            # For LRT, download all CSV and Excel files, no year filter
            df_mode = load_mode_delay_data(mode_name, pkg_id, extra_csv="TTC LRT Delays.csv",
                                           force_csv_only=False, skip_year_filter=True)
        else:
            df_mode = load_mode_delay_data(mode_name, pkg_id)
        if not df_mode.empty:
            all_dfs.append(df_mode)

    if not all_dfs:
        raise Exception("No data loaded for any mode.")

    df_delay = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Combined delay data: {len(df_delay)} records")

    # 2. Convert Date and extract Year, Month, Weekday, Hour
    df_delay['Date'] = pd.to_datetime(df_delay['Date'], errors='coerce')
    df_delay['Year'] = df_delay['Date'].dt.year
    df_delay['Month'] = df_delay['Date'].dt.month
    df_delay['Weekday'] = df_delay['Date'].dt.day_name()
    if 'Time' in df_delay.columns:
        def extract_hour(t):
            if pd.isna(t):
                return None
            match = re.search(r'(\d{1,2}):', str(t))
            return int(match.group(1)) if match else None
        df_delay['Hour'] = df_delay['Time'].apply(extract_hour)

    # 3. Download GTFS data
    routes, trips, stops = download_gtfs_schedule()

    generate_route_geometries()

    # 4. Prepare stop matching structures
    stop_matcher_data = prepare_stop_matcher(stops)

    # 5. Match locations to stops
    print("\n📍 Matching delay locations to GTFS stops...")
    match_results = df_delay['Location'].apply(
        lambda loc: pd.Series(match_location(loc, *stop_matcher_data))
    )
    match_results.columns = ['stop_name', 'stop_lat', 'stop_lon']
    df_delay = pd.concat([df_delay, match_results], axis=1)

    # 6. Extract numeric route (route_num) and compute active_in_2025
    df_delay['route_num'] = df_delay['Route'].astype(str).str.extract(r'^(\d+)')[0]
    routes_2025 = set(df_delay[df_delay['Year'] == 2025]['route_num'].dropna().unique())
    df_delay['active_in_2025'] = df_delay['route_num'].isin(routes_2025)

    # 7. Add route_long_name from routes.txt
    route_name_map = routes.set_index('route_id')['route_long_name'].to_dict()
    df_delay['route_long_name'] = df_delay['route_num'].map(route_name_map)

    # 8. Build route variants map and add Variants column (only for Bus routes > 6)
    route_variants_map = get_route_variants(trips)

    def get_variants(row):
        if pd.isna(row['route_num']):
            return None
        if row['Transit'] == 'Bus' and int(row['route_num']) > 6:
            return route_variants_map.get(row['route_num'], [])
        return None

    df_delay['Variants'] = df_delay.apply(get_variants, axis=1)
    # Convert list to JSON string for CSV storage
    df_delay['Variants'] = df_delay['Variants'].apply(lambda x: json.dumps(x) if x is not None else None)

    # 9. Drop duplicate rows
    initial_len = len(df_delay)
    df_delay = df_delay.drop_duplicates()
    print(f"🧹 Removed {initial_len - len(df_delay)} duplicate rows")
    print(f"📊 After deduplication: {len(df_delay)} records")

    # 10. Fill missing Incident with "General" (though the categorization already assigns 'Unknown' for missing codes)
    df_delay['Incident'] = df_delay['Incident'].fillna('General')
    df_delay['Incident_Category'] = df_delay['Incident_Category'].fillna('General')

    # 11. Save full enriched dataset
    full_output = os.path.join(output_dir, "full_delay_data.csv")
    df_delay.to_csv(full_output, index=False)
    print(f"\n💾 Saved full enriched data to {full_output}")

    # 12. Route analysis summary (group by numeric route and incident category)
    print("\n📈 Generating route analysis summary (by incident category)...")
    route_summary = df_delay.groupby(
        ['route_num', 'Year', 'Transit', 'Incident_Category']   # using Incident_Category
    ).agg(
        Delay_Count=('Min Delay', 'count'),
        Avg_Delay_Min=('Min Delay', 'mean'),
        Total_Delay_Min=('Min Delay', 'sum'),
        Unique_Vehicles=('Vehicle', 'nunique'),
        active_in_2025=('active_in_2025', 'first')
    ).reset_index()

    # Add route_long_name (constant per route_num)
    route_long = df_delay[['route_num', 'route_long_name']].drop_duplicates().set_index('route_num')
    route_summary['route_long_name'] = route_summary['route_num'].map(route_long['route_long_name'])

    # Rename route_num to Route for output
    route_summary = route_summary.rename(columns={'route_num': 'Route'})

    route_output = os.path.join(output_dir, "route_analysis.csv")
    route_summary.to_csv(route_output, index=False)
    print(f"💾 Saved route analysis to {route_output}")

    # 13. Location analysis summary (group by stop and incident category)
    print("\n📍 Generating location analysis summary (by incident category)...")
    location_summary = df_delay.groupby(
        ['stop_name', 'stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category']
    ).agg(
        Delay_Count=('Min Delay', 'count'),
        Avg_Delay_Min=('Min Delay', 'mean'),
        Total_Delay_Min=('Min Delay', 'sum')
    ).reset_index()

    location_output = os.path.join(output_dir, "location_analysis.csv")
    location_summary.to_csv(location_output, index=False)
    print(f"💾 Saved location analysis to {location_output}")

    print("\n" + "="*60)
    print("🎉 ALL PROCESSING COMPLETE")
    print("="*60)

    return df_delay, route_summary, location_summary


# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    full_df, route_df, location_df = load_all_ttc_delay_data()
    print("\n👀 First few rows of full data:")
    print(full_df.head())

📁 Output files will be saved to: assets\data

🚋 Processing Bus data (package: ttc-bus-delay-data)
📦 Found 20 resource(s)
  ⏭️  ttc-bus-delay-data-readme: no year found, skipping
  📄 Resource: ttc-bus-delay-data-2014 (year=2014, format=xlsx, datastore_active=False)
    ✅ Loaded 94217 records
  📄 Resource: ttc-bus-delay-data-2015 (year=2015, format=xlsx, datastore_active=False)
    ✅ Loaded 76510 records
  📄 Resource: ttc-bus-delay-data-2016 (year=2016, format=xlsx, datastore_active=False)
    ✅ Loaded 77088 records
  📄 Resource: ttc-bus-delay-data-2017 (year=2017, format=xlsx, datastore_active=False)
    ✅ Loaded 70303 records
  📄 Resource: ttc-bus-delay-data-2018 (year=2018, format=xlsx, datastore_active=False)
    ✅ Loaded 73927 records
  📄 Resource: ttc-bus-delay-data-2019 (year=2019, format=xlsx, datastore_active=False)
    ✅ Loaded 62376 records
  📄 Resource: ttc-bus-delay-data-2020 (year=2020, format=xlsx, datastore_active=False)
    ✅ Loaded 36151 records
  📄 Resource: ttc-bus-de

C:\Users\bains\AppData\Local\Temp\ipykernel_3660\1222410438.py:590: DtypeWarning: Columns (0: trip_short_name, 1: shape_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(StringIO(gtfs_data.get('trips.txt', '')))



🗺️ Generating route geometries from GTFS...
📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted trips.txt
✅ Extracted shapes.txt


C:\Users\bains\AppData\Local\Temp\ipykernel_3660\1222410438.py:745: DtypeWarning: Columns (0: trip_short_name, 1: shape_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))


⚠️ Shape ID nan not found for route 49S
✅ Route geometries saved to assets\data\route_geometries.json

📍 Matching delay locations to GTFS stops...


KeyboardInterrupt: 

In [48]:
import pandas as pd
full_df= pd.read_csv(r'C:\Users\bains\Downloads\TTC Routes and Schedules Data\assets\data\full_delay_data.csv') 
location_df= pd.read_csv(r'C:\Users\bains\Downloads\TTC Routes and Schedules Data\assets\data\location_analysis.csv') 
route_df= pd.read_csv(r'C:\Users\bains\Downloads\TTC Routes and Schedules Data\assets\data\route_analysis.csv') 

C:\Users\bains\AppData\Local\Temp\ipykernel_3660\2915492625.py:2: DtypeWarning: Columns (0: Route, 1: Route Name, 2: Incident_Code, 3: Variants) have mixed types. Specify dtype option on import or set low_memory=False.
  full_df= pd.read_csv(r'C:\Users\bains\Downloads\TTC Routes and Schedules Data\assets\data\full_delay_data.csv')


In [49]:
full_df[full_df['Route'].isnull()]

,Date,Route,Route Name,Time,Day,Location,Incident,Min Delay,Min Gap,Direction,...,Month,Weekday,Hour,stop_name,stop_lat,stop_lon,route_num,active_in_2025,route_long_name,Variants
479411,2020-08-09,NaN,NaN,18:56,Sunday,WARDEN STATION,Equipment / Mechanical,11.0,22.0,N,...,8.0,Sunday,18.0,Warden Station at Temporary Bus Bay 1,43.709817,-79.280007,NaN,False,NaN,NaN
479945,2020-08-16,NaN,NaN,17:54,Sunday,MCCOWAN AND SHEPPARD,Operations / Human Error,30.0,60.0,NaN,...,8.0,Sunday,17.0,Sheppard Ave East at McCowan Rd,43.789452,-79.259050,NaN,False,NaN,NaN
480549,2020-08-26,NaN,NaN,16:54,Wednesday,25 MUTUAL ST ( QUEEN A,General Delay / Other,1.0,1.0,NaN,...,8.0,Wednesday,16.0,Queen St East at Sumach St,43.656918,-79.358807,NaN,False,NaN,NaN
480645,2020-08-27,NaN,NaN,19:22,Thursday,WARDEN STATION,Equipment / Mechanical,7.0,14.0,S,...,8.0,Thursday,19.0,Warden Station at Temporary Bus Bay 1,43.709817,-79.280007,NaN,False,NaN,NaN
480724,2020-08-29,NaN,NaN,11:07,Saturday,WILSON TRACK AND STUCT,Collision / Roadblock,0.0,0.0,NaN,...,8.0,Saturday,11.0,949 Wilson Ave,43.728771,-79.474465,NaN,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1177169,2025-12-21,NaN,NaN,15:36,Sunday,DUNCAN SHOPS BRAKE SEC,Unknown,0.0,0.0,NaN,...,12.0,Sunday,15.0,Duncan Shops,43.673586,-79.417853,NaN,False,NaN,NaN
1177402,2025-12-24,NaN,NaN,08:58,Wednesday,MCBRIEN BUILDING,Unknown,0.0,0.0,NaN,...,12.0,Wednesday,8.0,1540 Kipling Ave - Richview Residence - Kiplin...,43.680574,-79.552944,NaN,False,NaN,NaN
1177458,2025-12-24,NaN,NaN,23:49,Wednesday,RUSSELL SUBSTATION,Unknown,0.0,0.0,NaN,...,12.0,Wednesday,23.0,NaN,NaN,NaN,NaN,False,NaN,NaN
1177651,2025-12-28,NaN,NaN,17:43,Sunday,WILSON TRACK AND STRUC,Unknown,0.0,0.0,NaN,...,12.0,Sunday,17.0,Transit Rd at Wilson Ave,43.734385,-79.452674,NaN,False,NaN,NaN


In [ ]:
import requests
import pandas as pd
import re
import os
import zipfile
import json
from io import BytesIO, StringIO
from datetime import datetime
from collections import defaultdict
from rapidfuzz import fuzz, process

# ----------------------------------------------------------------------
# Comprehensive incident categorization
# ----------------------------------------------------------------------
def _build_code_mapping():
    """Build dictionary mapping raw incident codes to standard categories."""
    mapping = {}

    # ---- Subway codes (EU, MU, PU, SU, TU) ----
    # Equipment / Mechanical (EU)
    eu_mech = [
        'EUAC', 'EUAL', 'EUATC', 'EUBK', 'EUBO', 'EUCA', 'EUCH', 'EUCO',
        'EUDO', 'EUECD', 'EUHV', 'EULT', 'EULV', 'EUNEA', 'EUNT', 'EUO',
        'EUPI', 'EUSC', 'EUTL', 'EUTM', 'EUTR', 'EUTRD', 'EUVA', 'EUVE', 'EUYRD'
    ]
    for code in eu_mech:
        mapping[code] = 'Equipment / Mechanical'

    mapping['EUCD'] = 'General Delay / Other'          # Consequential Delay
    mapping['EUME'] = 'Operations / Human Error'       # Maintenance Error
    mapping['EUOE'] = 'Operations / Human Error'       # Rail Cars & Shops Opr. Error
    mapping['EUOPO'] = 'Infrastructure / Track / Signals'

    # Miscellaneous (MU)
    mapping['MUD']   = 'Passenger / Security'
    mapping['MUDD']  = 'External / Environment'
    mapping['MUEC']  = 'Infrastructure / Track / Signals'
    mapping['MUESA'] = 'Operations / Human Error'
    mapping['MUFM']  = 'External / Environment'
    mapping['MUFS']  = 'External / Environment'
    mapping['MUGD']  = 'General Delay / Other'
    mapping['MUI']   = 'Passenger / Security'
    mapping['MUIE']  = 'Passenger / Security'
    mapping['MUIR']  = 'Passenger / Security'
    mapping['MUIRS'] = 'Passenger / Security'
    mapping['MUIS']  = 'Passenger / Security'
    mapping['MULD']  = 'Management / Administrative'
    mapping['MUNOA'] = 'Operations / Human Error'
    mapping['MUO']   = 'General Delay / Other'
    mapping['MUODC'] = 'Infrastructure / Track / Signals'
    mapping['MUPAA'] = 'Passenger / Security'
    mapping['MUPLA'] = 'External / Environment'
    mapping['MUPLB'] = 'External / Environment'
    mapping['MUPLC'] = 'External / Environment'
    mapping['MUPR1'] = 'Passenger / Security'
    mapping['MUSAN'] = 'Cleaning / Unsanitary'
    mapping['MUSC']  = 'Equipment / Mechanical'
    mapping['MUTD']  = 'Management / Administrative'
    mapping['MUTO']  = 'General Delay / Other'
    mapping['MUWEA'] = 'External / Environment'
    mapping['MUWR']  = 'Management / Administrative'

    # Infrastructure (PU)
    pu_infra = [
        'PUATC', 'PUCBI', 'PUCSC', 'PUCSS', 'PUDCS', 'PUMEL', 'PUMO',
        'PUOPO', 'PUSAC', 'PUSBE', 'PUSCA', 'PUSCR', 'PUSEA', 'PUSI',
        'PUSIO', 'PUSIS', 'PUSLC', 'PUSO', 'PUSRA', 'PUSSW', 'PUSTC',
        'PUSTP', 'PUSTS', 'PUSWZ', 'PUSZC', 'PUTCD', 'PUTD', 'PUTIJ',
        'PUTNT', 'PUTO', 'PUTOE', 'PUTR', 'PUTS', 'PUTSC', 'PUTSM',
        'PUTTC', 'PUTTP', 'PUTWZ'
    ]
    for code in pu_infra:
        mapping[code] = 'Infrastructure / Track / Signals'

    mapping['PUMST'] = 'Passenger / Security'
    mapping['PUTDN'] = 'External / Environment'
    mapping['PUTIS'] = 'External / Environment'
    mapping['PUSNT'] = 'General Delay / Other'

    # Security (SU)
    su_sec = [
        'SUAE', 'SUAP', 'SUBT', 'SUCOL', 'SUDP', 'SUEAS', 'SUG',
        'SUO', 'SUPOL', 'SUROB', 'SUSA', 'SUSP', 'SUUT'
    ]
    for code in su_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TU)
    mapping['TUATC'] = 'Operations / Human Error'
    mapping['TUCC']  = 'Operations / Human Error'
    mapping['TUDOE'] = 'Operations / Human Error'
    mapping['TUKEY'] = 'Operations / Human Error'
    mapping['TUML']  = 'Scheduling / Late Starts'
    mapping['TUMVS'] = 'Operations / Human Error'
    mapping['TUNIP'] = 'Operations / Human Error'
    mapping['TUNOA'] = 'Operations / Human Error'
    mapping['TUO']   = 'General Delay / Other'
    mapping['TUOPO'] = 'Operations / Human Error'
    mapping['TUOS']  = 'Operations / Human Error'
    mapping['TUS']   = 'Scheduling / Late Starts'
    mapping['TUSC']  = 'Operations / Human Error'
    mapping['TUSET'] = 'Operations / Human Error'
    mapping['TUST']  = 'External / Environment'
    mapping['TUSUP'] = 'Operations / Human Error'

    # ---- Streetcar codes (ER, MR, PR, SR, TR) ----
    # Equipment / Mechanical (ER)
    er_mech = [
        'ERAC', 'ERBO', 'ERCO', 'ERDB', 'ERDO', 'ERHV', 'ERLT', 'ERLV',
        'ERNEA', 'ERNT', 'ERO', 'ERPR', 'ERRA', 'ERTB', 'ERTC', 'ERTL',
        'ERTR', 'ERVE', 'ERWA', 'ERWS'
    ]
    for code in er_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['ERCD'] = 'General Delay / Other'
    mapping['ERME'] = 'Operations / Human Error'

    # Miscellaneous (MR)
    mapping['MRCL']  = 'Management / Administrative'
    mapping['MRD']   = 'Passenger / Security'
    mapping['MRDD']  = 'External / Environment'
    mapping['MREC']  = 'Infrastructure / Track / Signals'
    mapping['MRESA'] = 'Operations / Human Error'
    mapping['MRFS']  = 'External / Environment'
    mapping['MRIE']  = 'Passenger / Security'
    mapping['MRLD']  = 'Management / Administrative'
    mapping['MRNOA'] = 'Operations / Human Error'
    mapping['MRO']   = 'General Delay / Other'
    mapping['MRPAA'] = 'Passenger / Security'
    mapping['MRPLA'] = 'External / Environment'
    mapping['MRPLB'] = 'External / Environment'
    mapping['MRPLC'] = 'External / Environment'
    mapping['MRPR1'] = 'Passenger / Security'
    mapping['MRSAN'] = 'Cleaning / Unsanitary'
    mapping['MRSTM'] = 'Infrastructure / Track / Signals'
    mapping['MRTO']  = 'General Delay / Other'
    mapping['MRUI']  = 'Passenger / Security'
    mapping['MRUIR'] = 'Passenger / Security'
    mapping['MRWEA'] = 'External / Environment'

    # Infrastructure (PR)
    mapping['PREL']  = 'Infrastructure / Track / Signals'
    mapping['PRO']   = 'General Delay / Other'
    mapping['PRS']   = 'Infrastructure / Track / Signals'
    mapping['PRSA']  = 'Infrastructure / Track / Signals'
    mapping['PRSL']  = 'Infrastructure / Track / Signals'
    mapping['PRSO']  = 'Infrastructure / Track / Signals'
    mapping['PRSP']  = 'Infrastructure / Track / Signals'
    mapping['PRST']  = 'Passenger / Security'
    mapping['PRSW']  = 'Infrastructure / Track / Signals'
    mapping['PRTST'] = 'Infrastructure / Track / Signals'
    mapping['PRW']   = 'Infrastructure / Track / Signals'

    # Security (SR)
    sr_sec = [
        'SRAE', 'SRAP', 'SRBT', 'SRCOL', 'SRDP', 'SREAS', 'SRO',
        'SRSA', 'SRSP', 'SRUT'
    ]
    for code in sr_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TR)
    mapping['TRDOE'] = 'Operations / Human Error'
    mapping['TRNIP'] = 'Operations / Human Error'
    mapping['TRNOA'] = 'Operations / Human Error'
    mapping['TRO']   = 'General Delay / Other'
    mapping['TRSET'] = 'Operations / Human Error'
    mapping['TRST']  = 'External / Environment'
    mapping['TRTC']  = 'Operations / Human Error'

    # ---- LRT codes (EX, MX, PX, SX, TX) ----
    # Equipment / Mechanical (EX)
    ex_mech = [
        'EXAC', 'EXBK', 'EXBO', 'EXCB', 'EXCE', 'EXCO', 'EXDB', 'EXDO',
        'EXECD', 'EXGA', 'EXGF', 'EXHV', 'EXLT', 'EXNEA', 'EXNT', 'EXO',
        'EXOSC', 'EXSA', 'EXSE', 'EXTB', 'EXTM', 'EXTR', 'EXVC', 'EXVE',
        'EXWA', 'EXWM', 'EXWS', 'EXYRD'
    ]
    for code in ex_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['EXADD'] = 'Equipment / Mechanical'
    mapping['EXOE']  = 'Operations / Human Error'
    mapping['EXPD']  = 'Collision / Roadblock'
    mapping['EXPI']  = 'Collision / Roadblock'

    # Miscellaneous (MX)
    mapping['MXAFR'] = 'Infrastructure / Track / Signals'
    mapping['MXCL']  = 'Management / Administrative'
    mapping['MXCSA'] = 'Operations / Human Error'
    mapping['MXD']   = 'Passenger / Security'
    mapping['MXDD']  = 'External / Environment'
    mapping['MXESA'] = 'Operations / Human Error'
    mapping['MXFM']  = 'External / Environment'
    mapping['MXFS']  = 'External / Environment'
    mapping['MXGD']  = 'General Delay / Other'
    mapping['MXI']   = 'Passenger / Security'
    mapping['MXIC']  = 'Passenger / Security'
    mapping['MXIE']  = 'Passenger / Security'
    mapping['MXIR']  = 'Passenger / Security'
    mapping['MXIRS'] = 'Passenger / Security'
    mapping['MXIS']  = 'Passenger / Security'
    mapping['MXLDC'] = 'Management / Administrative'
    mapping['MXLDT'] = 'Management / Administrative'
    mapping['MXNCA'] = 'Operations / Human Error'
    mapping['MXNOA'] = 'Operations / Human Error'
    mapping['MXO']   = 'General Delay / Other'
    mapping['MXPAA'] = 'Passenger / Security'
    mapping['MXPF']  = 'Infrastructure / Track / Signals'
    mapping['MXPLA'] = 'External / Environment'
    mapping['MXPLB'] = 'External / Environment'
    mapping['MXPLC'] = 'External / Environment'
    mapping['MXPR']  = 'External / Environment'
    mapping['MXPR1'] = 'Passenger / Security'
    mapping['MXPU']  = 'Operations / Human Error'
    mapping['MXSAN'] = 'Cleaning / Unsanitary'
    mapping['MXTD']  = 'Management / Administrative'
    mapping['MXTO']  = 'General Delay / Other'
    mapping['MXUS']  = 'Scheduling / Late Starts'
    mapping['MXWEA'] = 'External / Environment'
    mapping['MXWR']  = 'Management / Administrative'

    # Infrastructure (PX)
    mapping['PXATC'] = 'Infrastructure / Track / Signals'
    mapping['PXDCS'] = 'Infrastructure / Track / Signals'
    mapping['PXEAS'] = 'Infrastructure / Track / Signals'
    mapping['PXEME'] = 'Operations / Human Error'
    mapping['PXEO']  = 'General Delay / Other'
    mapping['PXMEL'] = 'Infrastructure / Track / Signals'
    mapping['PXMO']  = 'General Delay / Other'
    mapping['PXMST'] = 'Passenger / Security'
    mapping['PXOV']  = 'Infrastructure / Track / Signals'
    mapping['PXSAC'] = 'Infrastructure / Track / Signals'
    mapping['PXSBE'] = 'Infrastructure / Track / Signals'
    mapping['PXSCA'] = 'Infrastructure / Track / Signals'
    mapping['PXSCR'] = 'Infrastructure / Track / Signals'
    mapping['PXSI']  = 'Infrastructure / Track / Signals'
    mapping['PXSIS'] = 'Infrastructure / Track / Signals'
    mapping['PXSNT'] = 'General Delay / Other'
    mapping['PXSO']  = 'General Delay / Other'
    mapping['PXSRA'] = 'Infrastructure / Track / Signals'
    mapping['PXSTP'] = 'Infrastructure / Track / Signals'
    mapping['PXSW']  = 'Infrastructure / Track / Signals'
    mapping['PXTD']  = 'Infrastructure / Track / Signals'
    mapping['PXTDN'] = 'External / Environment'
    mapping['PXTIS'] = 'External / Environment'
    mapping['PXTR']  = 'Infrastructure / Track / Signals'
    mapping['PXTS']  = 'Infrastructure / Track / Signals'
    mapping['PXW']   = 'Infrastructure / Track / Signals'
    mapping['PXWZ']  = 'Infrastructure / Track / Signals'

    # Security (SX)
    sx_sec = [
        'SXAE', 'SXAM', 'SXAP', 'SXAX', 'SXBT', 'SXCOL', 'SXDP',
        'SXEAS', 'SXG', 'SXO', 'SXPOL', 'SXROB', 'SXSA', 'SXSP', 'SXUEG'
    ]
    for code in sx_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TX)
    mapping['TXATC'] = 'Operations / Human Error'
    mapping['TXCC']  = 'Operations / Human Error'
    mapping['TXDOE'] = 'Operations / Human Error'
    mapping['TXLF']  = 'Scheduling / Late Starts'
    mapping['TXML']  = 'Scheduling / Late Starts'
    mapping['TXMVS'] = 'Operations / Human Error'
    mapping['TXNCA'] = 'Operations / Human Error'
    mapping['TXNIP'] = 'Operations / Human Error'
    mapping['TXNOA'] = 'Operations / Human Error'
    mapping['TXO']   = 'General Delay / Other'
    mapping['TXOI']  = 'Passenger / Security'
    mapping['TXOS']  = 'Operations / Human Error'
    mapping['TXOVS'] = 'Operations / Human Error'
    mapping['TXPD']  = 'Collision / Roadblock'
    mapping['TXPI']  = 'Collision / Roadblock'
    mapping['TXS']   = 'Scheduling / Late Starts'
    mapping['TXST']  = 'External / Environment'
    mapping['TXSUP'] = 'Operations / Human Error'
    mapping['TXSV']  = 'Operations / Human Error'

    return mapping

# Global mapping dictionary (built once)
_INCIDENT_CODE_MAP = _build_code_mapping()

def categorize_incident(code):
    """
    Convert an incident code or description into a standard category.
    Returns one of ten categories:
        - Equipment / Mechanical
        - Operations / Human Error
        - Infrastructure / Track / Signals
        - Passenger / Security
        - External / Environment
        - Scheduling / Late Starts
        - Collision / Roadblock
        - Cleaning / Unsanitary
        - Management / Administrative
        - General Delay / Other
    """
    if pd.isna(code):
        return 'Unknown'
    s = str(code).strip()
    if not s:
        return 'Unknown'

    # Step 1: If it looks like a pure code (all caps, 2-6 letters), try the code map
    if re.match(r'^[A-Z]{2,6}$', s):
        return _INCIDENT_CODE_MAP.get(s, 'General Delay / Other')

    # Step 2: Otherwise, treat as descriptive text and apply patterns
    s_lower = s.lower()

    # Priority order: more specific patterns first
    patterns = [
        (r'late (leaving|entering)|unable to maintain schedule|mainline storage', 'Scheduling / Late Starts'),
        (r'collision|road ?block', 'Collision / Roadblock'),
        (r'clean|unsanitary|disinfection', 'Cleaning / Unsanitary'),
        (r'management|clerk|training|labour dispute|work refusal', 'Management / Administrative'),
        (r'weather|ice|snow|fire|debris|force majeure|storm', 'External / Environment'),
        (r'mechanical|equipment|brakes|door.*faulty|hvac|propulsion', 'Equipment / Mechanical'),
        (r'operations?.*operator|signal violation|overshot|overspeed|not in position|supervisory', 'Operations / Human Error'),
        (r'passenger|security|assault|disorderly|bomb|alarm|unauthorized|injur', 'Passenger / Security'),
        (r'infrastructure|track|signal|power|escalator|elevator|switch|rail|debris.*controllable', 'Infrastructure / Track / Signals'),
        (r'general delay|consequential|no trouble|other', 'General Delay / Other'),
    ]

    for pattern, category in patterns:
        if re.search(pattern, s_lower):
            return category

    # Fallback
    return 'General Delay / Other'


def clean_and_standardize(df):
    """Standardize column names and enforce a uniform schema."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    column_mappings = {
        'Date': ['Report Date', 'Date', 'Incident Date', 'Date & Time'],
        'Time': ['Time', 'Incident Time'],
        'Day': ['Day'],
        'Location': ['Location', 'Station', 'Station Name', 'Stop', 'Stop Name'],
        'Incident': ['Incident', 'Code', 'Description'],
        'Min Delay': ['Min Delay', 'Delay', 'Delay Minutes', 'Delay_Minutes'],
        'Min Gap': ['Min Gap', 'Gap', 'Gap Minutes', 'Gap_Minutes'],
        'Route': ['Route', 'Route Number', 'Route No', 'Route_ID'],
        'Line': ['Line'],
        'Direction': ['Direction', 'Bound'],
        'Vehicle': ['Vehicle', 'Vehicle Number', 'Vehicle_No']
    }

    reverse_mapping = {}
    for std_name, possible in column_mappings.items():
        for name in possible:
            reverse_mapping[name] = std_name

    rename_dict = {col: reverse_mapping[col] for col in df.columns if col in reverse_mapping}
    df = df.rename(columns=rename_dict)

    # Handle Line column -> Route, Route Name
    if 'Line' in df.columns and 'Route' not in df.columns:
        def extract_route_info(line_val):
            if pd.isna(line_val):
                return pd.Series([None, None])
            line_str = str(line_val).strip()
            match = re.match(r'^(\d+)(?:\s+(.+))?$', line_str)
            if match:
                return pd.Series([match.group(1), match.group(2) if match.group(2) else ''])
            match_digits = re.match(r'^(\d+)$', line_str)
            if match_digits:
                return pd.Series([match_digits.group(1), ''])
            return pd.Series([line_str, ''])
        df[['Route', 'Route Name']] = df['Line'].apply(extract_route_info)

    if 'Route' in df.columns and 'Route Name' not in df.columns:
        df['Route Name'] = ''

    required_columns = [
        'Date', 'Route', 'Route Name', 'Time', 'Day', 'Location',
        'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle',
        'Incident_Original'                    # <-- changed from Incident_Code
    ]
    for col in required_columns:
        if col not in df.columns:
            df[col] = None

    df = df[required_columns]
    return df


# ----------------------------------------------------------------------
# Download and merge delay data for a single mode
# ----------------------------------------------------------------------
def load_mode_delay_data(mode_name, package_id, extra_csv=None, force_csv_only=False, skip_year_filter=False):
    """Downloads all resources for a given transit mode, cleans and returns a DataFrame.

    If skip_year_filter=True, no year extraction or filtering is performed – all resources
    with allowed formats are downloaded regardless of filename year.
    """
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    current_year = datetime.now().year
    print(f"\n🚋 Processing {mode_name} data (package: {package_id})")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception(f"CKAN API request failed for {mode_name}")

    resources = data["result"]["resources"]
    print(f"📦 Found {len(resources)} resource(s)")

    year_pattern = re.compile(r"(19|20)\d{2}")
    allowed_formats = {'csv'} if force_csv_only else {'csv', 'xlsx', 'xls'}

    mode_dfs = []

    for res in resources:
        res_name = res.get("name", "Unnamed")
        datastore_active = res.get("datastore_active", False)

        # --- Year extraction and filtering (skip if skip_year_filter=True) ---
        if not skip_year_filter:
            year_match = year_pattern.search(res_name)
            if not year_match:
                print(f"  ⏭️  {res_name}: no year found, skipping")
                continue
            year = int(year_match.group(0))
            if year < 2014:
                print(f"  ⏭️  {res_name}: year {year} < 2014, skipping")
                continue
        else:
            year = None   # No year information

        # Determine format
        if datastore_active:
            fmt = 'csv'
        else:
            fmt = res.get("format", "").lower()
            if not fmt:
                url = res.get("url", "")
                if url.endswith('.csv'):
                    fmt = 'csv'
                elif url.endswith(('.xlsx', '.xls')):
                    fmt = 'xlsx'
                else:
                    fmt = 'unknown'

        if fmt not in allowed_formats:
            print(f"  ⏭️  {res_name}: format '{fmt}' not in {allowed_formats}, skipping")
            continue

        # Apply year‑based format filter only if not skipped
        if not skip_year_filter and not force_csv_only:
            if year >= current_year-1 and fmt != 'csv':
                print(f"  ⏭️  {res_name}: skipping XLSX for latest year, only CSV kept")
                continue
            elif year < current_year-1 and fmt not in ('xlsx', 'xls'):
                print(f"  ⏭️  {res_name}: skipping CSV for year < {current_year}, only XLSX kept")
                continue

        # Print info, including year if available
        if year:
            print(f"  📄 Resource: {res_name} (year={year}, format={fmt}, datastore_active={datastore_active})")
        else:
            print(f"  📄 Resource: {res_name} (format={fmt}, datastore_active={datastore_active})")

        try:
            if datastore_active:
                dump_url = f"{base_url}/datastore/dump/{res['id']}"
                dump_resp = requests.get(dump_url)
                dump_resp.raise_for_status()
                df = pd.read_csv(StringIO(dump_resp.text))
                if 'Incident' in df.columns:
                    df['Incident_Original'] = df['Incident']   # <-- preserve original
            else:
                file_url = res["url"]
                file_resp = requests.get(file_url)
                file_resp.raise_for_status()

                if fmt in ('xlsx', 'xls'):
                    excel_data = pd.read_excel(BytesIO(file_resp.content), sheet_name=None)
                    sheet_dfs = []
                    for sheet_name, sheet_df in excel_data.items():
                        if not sheet_df.empty:
                            if 'Incident' in sheet_df.columns:
                                sheet_df['Incident_Original'] = sheet_df['Incident']   # <-- preserve original
                            sheet_df = clean_and_standardize(sheet_df)
                            sheet_dfs.append(sheet_df)
                    if sheet_dfs:
                        df = pd.concat(sheet_dfs, ignore_index=True)
                    else:
                        print(f"    ⚠️ No data in any sheet, skipping")
                        continue
                else:
                    df = pd.read_csv(StringIO(file_resp.text))
                    if 'Incident' in df.columns:
                        df['Incident_Original'] = df['Incident']   # <-- preserve original

            # Clean and standardize (the cleaned df now includes Incident_Original)
            df = clean_and_standardize(df)

            # Apply incident categorization directly to the raw value
            df['Incident'] = df['Incident_Original'].apply(categorize_incident)
            # Also set Incident_Category to the same value (used in summaries)
            df['Incident_Category'] = df['Incident']
            df['Transit'] = mode_name

            mode_dfs.append(df)
            print(f"    ✅ Loaded {len(df)} records")

        except Exception as e:
            print(f"    ❌ Error: {e}")
            continue

    if extra_csv and os.path.isfile(extra_csv):
        print(f"\n  📄 Extra local file: {extra_csv}")
        try:
            df_extra = pd.read_csv(extra_csv)
            if 'Incident' in df_extra.columns:
                df_extra['Incident_Original'] = df_extra['Incident']   # <-- preserve original
            df_extra = clean_and_standardize(df_extra)
            df_extra['Incident'] = df_extra['Incident_Original'].apply(categorize_incident)
            df_extra['Incident_Category'] = df_extra['Incident']
            df_extra['Transit'] = mode_name
            mode_dfs.append(df_extra)
            print(f"    ✅ Loaded {len(df_extra)} records from extra file")
        except Exception as e:
            print(f"    ❌ Error reading extra file {extra_csv}: {e}")
    elif extra_csv:
        print(f"\n  ⏭️ Extra file {extra_csv} not found, skipping")

    if not mode_dfs:
        print(f"⚠️ No valid data loaded for {mode_name}")
        return pd.DataFrame()

    mode_combined = pd.concat(mode_dfs, ignore_index=True)
    print(f"✅ {mode_name} total records: {len(mode_combined)}")
    return mode_combined


# ----------------------------------------------------------------------
# Download GTFS static data (routes, trips, stops)
# ----------------------------------------------------------------------
def download_gtfs_schedule():
    """Download and extract routes.txt, trips.txt, stops.txt from the merged GTFS package."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    print("\n🗺️ Downloading GTFS schedule data...")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource (complete GTFS)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract required files into memory
    gtfs_data = {}
    required_files = ['routes.txt', 'trips.txt', 'stops.txt']
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")

    # Convert to DataFrames
    routes = pd.read_csv(StringIO(gtfs_data.get('routes.txt', '')))
    trips = pd.read_csv(StringIO(gtfs_data.get('trips.txt', '')))
    stops = pd.read_csv(StringIO(gtfs_data.get('stops.txt', '')))

    return routes, trips, stops


# ----------------------------------------------------------------------
# Prepare stop matching data structures
# ----------------------------------------------------------------------
def prepare_stop_matcher(stops_df):
    """Build normalized stop names and inverted index for fast matching."""
    def normalize_stop(name):
        if pd.isna(name):
            return ''
        s = name.lower()
        s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    stops = stops_df.copy()
    stops['norm_stop'] = stops['stop_name'].apply(normalize_stop)
    stops['stop_words'] = stops['norm_stop'].apply(lambda x: set(x.split()))

    stop_names = stops['stop_name'].tolist()
    stop_norms = stops['norm_stop'].tolist()
    stop_words_list = stops['stop_words'].tolist()
    stop_lats = stops['stop_lat'].tolist()
    stop_lons = stops['stop_lon'].tolist()

    word_to_indices = defaultdict(set)
    for idx, words in enumerate(stop_words_list):
        for word in words:
            word_to_indices[word].add(idx)

    return (stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices)


# ----------------------------------------------------------------------
# Clean a delay location string
# ----------------------------------------------------------------------
def clean_location(loc):
    if pd.isna(loc):
        return ''
    s = loc.lower()
    s = re.sub(r'\(.*?\)', '', s)       # remove parenthetical notes
    s = re.sub(r'[^\w\s]', ' ', s)      # punctuation -> space
    s = re.sub(r'\s+', ' ', s).strip()
    # Expand common direction abbreviations
    s = re.sub(r'\bw\b', ' west', s)
    s = re.sub(r'\be\b', ' east', s)
    s = re.sub(r'\bn\b', ' north', s)
    s = re.sub(r'\bs\b', ' south', s)
    return s


# ----------------------------------------------------------------------
# Match a delay location to a stop (using pre‑built structures)
# ----------------------------------------------------------------------
def match_location(loc, stop_names, stop_norms, stop_words_list, stop_lats, stop_lons, word_to_indices):
    if pd.isna(loc):
        return None, None, None

    cleaned = clean_location(loc)
    words = set(cleaned.split())

    # Find candidate stops that contain *any* of the words
    candidates = set()
    for w in words:
        candidates.update(word_to_indices.get(w, set()))
    if not candidates:
        return None, None, None

    # 1. Exact subset match (all words appear)
    for idx in candidates:
        if words.issubset(stop_words_list[idx]):
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 2. Substring match (cleaned location appears in normalized stop name)
    for idx in candidates:
        if cleaned in stop_norms[idx]:
            return stop_names[idx], stop_lats[idx], stop_lons[idx]

    # 3. Fuzzy match using rapidfuzz
    cand_indices = list(candidates)
    cand_norms = [stop_norms[i] for i in cand_indices]
    result = process.extractOne(
        cleaned,
        cand_norms,
        scorer=fuzz.token_set_ratio,
        score_cutoff=60,
        processor=None
    )
    if result:
        best_pos = cand_norms.index(result[0])
        best_idx = cand_indices[best_pos]
        return stop_names[best_idx], stop_lats[best_idx], stop_lons[best_idx]

    return None, None, None



def generate_route_geometries():
    """
    Downloads trips.txt and shapes.txt from the merged GTFS package,
    and generates a JSON file mapping route+short_name combinations
    to their shape geometries (list of [lat, lon] points).
    The file is saved as assets/data/route_geometries.json.
    """
    import json
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "route_geometries.json")

    print("\n🗺️ Generating route geometries from GTFS...")

    # 1. Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # 2. Find the ZIP resource (non-datastore, format=ZIP)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # 3. Extract trips.txt and shapes.txt into memory
    required_files = ['trips.txt', 'shapes.txt']
    gtfs_data = {}
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")
                return

    # 4. Read trips.txt
    trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))
    # Keep only necessary columns: route_id, trip_short_name, shape_id
    trips = trips[['route_id', 'trip_short_name', 'shape_id']].drop_duplicates()
    # Convert to string and fill missing short names
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()

    # 5. Read shapes.txt
    shapes = pd.read_csv(StringIO(gtfs_data['shapes.txt']))
    shapes = shapes[['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']]
    shapes = shapes.sort_values(['shape_id', 'shape_pt_sequence'])

    # 6. Build geometry per shape_id
    shape_geometries = {}
    for shape_id, group in shapes.groupby('shape_id'):
        # Create list of [lat, lon] pairs in order
        coords = group[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
        shape_geometries[shape_id] = coords

    # 7. Merge trips with shape geometries
    # For each (route_id, trip_short_name) we need a geometry.
    # There might be multiple trips with same route+short_name but different shape_id.
    # We'll take the first shape_id for each combination (assuming consistency).
    # Alternatively, we could keep all, but the user likely wants one geometry per route variant.
    route_geometries = {}
    # Group trips by (route_id, trip_short_name) and take first shape_id
    for (route_id, short_name), group in trips.groupby(['route_id', 'trip_short_name']):
        shape_id = group.iloc[0]['shape_id']  # first shape_id
        if shape_id in shape_geometries:
            key = route_id + short_name  # e.g., "100A"
            route_geometries[key] = shape_geometries[shape_id]
        else:
            print(f"⚠️ Shape ID {shape_id} not found for route {key}")

    # 8. Save to JSON
    with open(output_file, 'w') as f:
        json.dump(route_geometries, f, indent=2)
    print(f"✅ Route geometries saved to {output_file}")
    return route_geometries



# ----------------------------------------------------------------------
# Build route variants map from trips.txt
# ----------------------------------------------------------------------
def get_route_variants(trips_df):
    """Return dict {route_id: list_of_full_route_names} e.g. {'129': ['129A','129B','129C']}"""
    trips = trips_df.copy()
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()
    # Keep only rows with a non‑empty short name
    trips = trips[trips['trip_short_name'] != '']
    variants = trips.groupby('route_id')['trip_short_name'].apply(list).to_dict()
    route_variants = {}
    for route, short_names in variants.items():
        route_variants[route] = sorted([f"{route}{sn}" for sn in short_names if sn])
    return route_variants


# ----------------------------------------------------------------------
# Main: load all modes, merge, enrich, and produce summaries
# ----------------------------------------------------------------------
def load_all_ttc_delay_data():
    """Complete pipeline: download all delay data, GTFS, enrich, and output summaries."""

    # Define output directory
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output files will be saved to: {output_dir}")

    # 1. Download delay data for all modes
    modes = [
        ("Bus", "ttc-bus-delay-data"),
        ("Streetcar", "ttc-streetcar-delay-data"),
        ("Subway", "ttc-subway-delay-data"),
        ("LRT", "ttc-lrt-delay-data")
    ]

    all_dfs = []
    for mode_name, pkg_id in modes:
        if mode_name == "LRT":
            # For LRT, download all CSV and Excel files, no year filter
            df_mode = load_mode_delay_data(mode_name, pkg_id, extra_csv="TTC LRT Delays.csv",
                                           force_csv_only=False, skip_year_filter=True)
        else:
            df_mode = load_mode_delay_data(mode_name, pkg_id)
        if not df_mode.empty:
            all_dfs.append(df_mode)

    if not all_dfs:
        raise Exception("No data loaded for any mode.")

    df_delay = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Combined delay data: {len(df_delay)} records")

    # 2. Convert Date and extract Year, Month, Weekday, Hour
    df_delay['Date'] = pd.to_datetime(df_delay['Date'], errors='coerce')
    df_delay['Year'] = df_delay['Date'].dt.year
    df_delay['Month'] = df_delay['Date'].dt.month
    df_delay['Weekday'] = df_delay['Date'].dt.day_name()
    if 'Time' in df_delay.columns:
        def extract_hour(t):
            if pd.isna(t):
                return None
            match = re.search(r'(\d{1,2}):', str(t))
            return int(match.group(1)) if match else None
        df_delay['Hour'] = df_delay['Time'].apply(extract_hour)

    # 3. Download GTFS data
    routes, trips, stops = download_gtfs_schedule()

    generate_route_geometries()

    # 4. Prepare stop matching structures
    stop_matcher_data = prepare_stop_matcher(stops)

    # 5. Match locations to stops
    print("\n📍 Matching delay locations to GTFS stops...")
    match_results = df_delay['Location'].apply(
        lambda loc: pd.Series(match_location(loc, *stop_matcher_data))
    )
    match_results.columns = ['stop_name', 'stop_lat', 'stop_lon']
    df_delay = pd.concat([df_delay, match_results], axis=1)

    # 6. Extract numeric route (route_num) and compute active_in_2025
    df_delay['route_num'] = df_delay['Route'].astype(str).str.extract(r'^(\d+)')[0]
    routes_2025 = set(df_delay[df_delay['Year'] == 2025]['route_num'].dropna().unique())
    df_delay['active_in_2025'] = df_delay['route_num'].isin(routes_2025)

    # 7. Add route_long_name from routes.txt
    route_name_map = routes.set_index('route_id')['route_long_name'].to_dict()
    df_delay['route_long_name'] = df_delay['route_num'].map(route_name_map)

    # 8. Build route variants map and add Variants column (only for Bus routes > 6)
    route_variants_map = get_route_variants(trips)

    def get_variants(row):
        if pd.isna(row['route_num']):
            return None
        if row['Transit'] == 'Bus' and int(row['route_num']) > 6:
            return route_variants_map.get(row['route_num'], [])
        return None

    df_delay['Variants'] = df_delay.apply(get_variants, axis=1)
    # Convert list to JSON string for CSV storage
    df_delay['Variants'] = df_delay['Variants'].apply(lambda x: json.dumps(x) if x is not None else None)

    # 9. Drop duplicate rows
    initial_len = len(df_delay)
    df_delay = df_delay.drop_duplicates()
    print(f"🧹 Removed {initial_len - len(df_delay)} duplicate rows")
    print(f"📊 After deduplication: {len(df_delay)} records")

    # 10. Fill missing Incident with "General" (though the categorization already assigns 'Unknown' for missing codes)
    df_delay['Incident'] = df_delay['Incident'].fillna('General')
    df_delay['Incident_Category'] = df_delay['Incident_Category'].fillna('General')

    # 11. Save full enriched dataset
    full_output = os.path.join(output_dir, "full_delay_data.csv")
    df_delay.to_csv(full_output, index=False)
    print(f"\n💾 Saved full enriched data to {full_output}")

    # 12. Route analysis summary (group by numeric route and incident category)
    print("\n📈 Generating route analysis summary (by incident category)...")
    route_summary = df_delay.groupby(
        ['route_num', 'Year', 'Transit', 'Incident_Category']   # using Incident_Category
    ).agg(
        Delay_Count=('Min Delay', 'count'),
        Avg_Delay_Min=('Min Delay', 'mean'),
        Total_Delay_Min=('Min Delay', 'sum'),
        Unique_Vehicles=('Vehicle', 'nunique'),
        active_in_2025=('active_in_2025', 'first')
    ).reset_index()

    # Add route_long_name (constant per route_num)
    route_long = df_delay[['route_num', 'route_long_name']].drop_duplicates().set_index('route_num')
    route_summary['route_long_name'] = route_summary['route_num'].map(route_long['route_long_name'])

    # Rename route_num to Route for output
    route_summary = route_summary.rename(columns={'route_num': 'Route'})

    route_output = os.path.join(output_dir, "route_analysis.csv")
    route_summary.to_csv(route_output, index=False)
    print(f"💾 Saved route analysis to {route_output}")

    # 13. Location analysis summary (group by stop and incident category)
    print("\n📍 Generating location analysis summary (by incident category)...")
    location_summary = df_delay.groupby(
        ['stop_name', 'stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category']
    ).agg(
        Delay_Count=('Min Delay', 'count'),
        Avg_Delay_Min=('Min Delay', 'mean'),
        Total_Delay_Min=('Min Delay', 'sum')
    ).reset_index()

    location_output = os.path.join(output_dir, "location_analysis.csv")
    location_summary.to_csv(location_output, index=False)
    print(f"💾 Saved location analysis to {location_output}")

    print("\n" + "="*60)
    print("🎉 ALL PROCESSING COMPLETE")
    print("="*60)

    return df_delay, route_summary, location_summary


# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    full_df, route_df, location_df = load_all_ttc_delay_data()
    print("\n👀 First few rows of full data:")
    print(full_df.head())

🚌 Fetching bus delay package metadata...
📦 Found 20 resource(s)
  📄 Downloading: ttc-bus-delay-data-readme (format=xlsx)
    ✅ Added 10 records
  📄 Downloading: ttc-bus-delay-data-2014 (format=xlsx)
    ✅ Added 94217 records
  📄 Downloading: ttc-bus-delay-data-2015 (format=xlsx)
    ✅ Added 76510 records
  📄 Downloading: ttc-bus-delay-data-2016 (format=xlsx)
    ✅ Added 77088 records
  📄 Downloading: ttc-bus-delay-data-2017 (format=xlsx)
    ✅ Added 70303 records
  📄 Downloading: ttc-bus-delay-data-2018 (format=xlsx)
    ✅ Added 73927 records
  📄 Downloading: ttc-bus-delay-data-2019 (format=xlsx)
    ✅ Added 62376 records
  📄 Downloading: ttc-bus-delay-data-2020 (format=xlsx)
    ✅ Added 36151 records
  📄 Downloading: ttc-bus-delay-data-2021 (format=xlsx)
    ✅ Added 42269 records
  📄 Downloading: ttc-bus-delay-data-2022 (format=xlsx)
    ✅ Added 58707 records
  📄 Downloading: ttc-bus-delay-data-2023 (format=xlsx)
    ✅ Added 56207 records
  📄 Downloading: ttc-bus-delay-data-2024 (form

In [16]:
import folium
import geopandas as gpd

# 1. Load the cleaned GeoJSON file
geojson_path = "assets/data/gtawards.geojson"
gdf = gpd.read_file(geojson_path)

# 2. Compute the map center (average of all ward centroids)
center_lat = gdf.geometry.centroid.y.mean()
center_lon = gdf.geometry.centroid.x.mean()

# 3. Create a Folium map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,                # Adjust zoom level as needed
    tiles="CartoDB positron"       # Light, clean basemap
)

# 4. Add each ward as a GeoJson layer with a popup
for _, row in gdf.iterrows():
    # Convert the Shapely geometry to GeoJSON format
    geo_json = row.geometry.__geo_interface__

    folium.GeoJson(
        geo_json,
        style_function=lambda x: {
            'fillColor': 'lightblue',
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(row['AREA_NAME'], parse_html=False)
    ).add_to(m)

# 5. Save the map as an HTML file
output_file = "assets/data/wards_map.html"
m.save(output_file)

print(f"✅ Map saved to {output_file}")

C:\Users\bains\AppData\Local\Temp\ipykernel_3660\149196161.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = gdf.geometry.centroid.y.mean()
C:\Users\bains\AppData\Local\Temp\ipykernel_3660\149196161.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = gdf.geometry.centroid.x.mean()


✅ Map saved to assets/data/wards_map.html


In [19]:
import pandas as pd
import geopandas as gpd
import folium

# -------------------------------
# 1. Load ward boundaries
# -------------------------------
wards_gdf = gpd.read_file("assets/data/gtawards.geojson")
# Ensure CRS is WGS84 (lat/lon)
if wards_gdf.crs is None:
    wards_gdf.set_crs(epsg=4326, inplace=True)
else:
    wards_gdf = wards_gdf.to_crs(epsg=4326)

# -------------------------------
# 2. Load location analysis data
# -------------------------------
loc_df = pd.read_csv("assets/data/location_analysis.csv")

# Drop rows with missing coordinates
loc_df = loc_df.dropna(subset=['stop_lat', 'stop_lon'])

# Create a GeoDataFrame from stop points
points_gdf = gpd.GeoDataFrame(
    loc_df,
    geometry=gpd.points_from_xy(loc_df['stop_lon'], loc_df['stop_lat']),
    crs="EPSG:4326"
)

# -------------------------------
# 3. Spatial join: assign ward to each point
# -------------------------------
# Use 'within' or 'intersects' to find which polygon contains each point
joined = gpd.sjoin(points_gdf, wards_gdf[['AREA_NAME', 'geometry']], how='left', predicate='within')

# Some points may fall outside any ward; we'll drop them for the choropleth
joined = joined.dropna(subset=['AREA_NAME'])

# -------------------------------
# 4. Aggregate incident counts per ward
# -------------------------------
# Delay_Count is already the number of incidents at that stop location.
# Sum it per ward.
ward_stats = joined.groupby('AREA_NAME')['Delay_Count'].sum().reset_index()
ward_stats.columns = ['AREA_NAME', 'total_incidents']

# Merge back with ward geometries for mapping
ward_map = wards_gdf.merge(ward_stats, on='AREA_NAME', how='left')
ward_map['total_incidents'] = ward_map['total_incidents'].fillna(0)  # wards with no incidents get 0

# -------------------------------
# 5. Create Folium choropleth map
# -------------------------------
# Calculate map center (average of all ward centroids)
center_lat = wards_gdf.geometry.centroid.y.mean()
center_lon = wards_gdf.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='CartoDB positron')

# Add choropleth layer
folium.Choropleth(
    geo_data=ward_map.to_json(),
    name='Incidents per Ward',
    data=ward_map,
    columns=['AREA_NAME', 'total_incidents'],
    key_on='feature.properties.AREA_NAME',
    fill_color='YlOrRd',          # yellow-orange-red color scale
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name='Total Incidents'
).add_to(m)

# Optionally add tooltips to show ward name and incident count on hover
folium.GeoJsonTooltip(fields=['AREA_NAME', 'total_incidents'],
                      aliases=['Ward:', 'Incidents:'],
                      localize=True).add_to(
    folium.GeoJson(
        ward_map.to_json(),
        style_function=lambda x: {'fillOpacity': 0, 'color': 'black', 'weight': 0.5}
    )
)

# -------------------------------
# 6. Save the map
# -------------------------------
output_file = "assets/data/incidents_by_ward.html"
m.save(output_file)
print(f"✅ Choropleth map saved to {output_file}")

C:\Users\bains\AppData\Local\Temp\ipykernel_3660\416827975.py:55: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = wards_gdf.geometry.centroid.y.mean()
C:\Users\bains\AppData\Local\Temp\ipykernel_3660\416827975.py:56: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = wards_gdf.geometry.centroid.x.mean()


✅ Choropleth map saved to assets/data/incidents_by_ward.html


In [18]:
import pandas as pd
import geopandas as gpd
import folium

# -------------------------------
# 1. Load neighborhood boundaries
# -------------------------------
neighborhoods_gdf = gpd.read_file("assets/data/toronto.geojson")
# Ensure CRS is WGS84 (lat/lon)
if neighborhoods_gdf.crs is None:
    neighborhoods_gdf.set_crs(epsg=4326, inplace=True)
else:
    neighborhoods_gdf = neighborhoods_gdf.to_crs(epsg=4326)

# Check available property columns (optional)
print("Neighborhood columns:", neighborhoods_gdf.columns.tolist())
# We'll use 'AREA_NAME' for labeling and aggregation.

# -------------------------------
# 2. Load location analysis data
# -------------------------------
loc_df = pd.read_csv("assets/data/location_analysis.csv")

# Drop rows with missing coordinates
loc_df = loc_df.dropna(subset=['stop_lat', 'stop_lon'])

# Create a GeoDataFrame from stop points
points_gdf = gpd.GeoDataFrame(
    loc_df,
    geometry=gpd.points_from_xy(loc_df['stop_lon'], loc_df['stop_lat']),
    crs="EPSG:4326"
)

# -------------------------------
# 3. Spatial join: assign neighborhood to each point
# -------------------------------
# Use 'within' to find which polygon contains each point
joined = gpd.sjoin(points_gdf, neighborhoods_gdf[['AREA_NAME', 'geometry']], how='left', predicate='within')

# Some points may fall outside any neighborhood; we'll drop them for the choropleth
joined = joined.dropna(subset=['AREA_NAME'])

# -------------------------------
# 4. Aggregate incident counts per neighborhood
# -------------------------------
# Delay_Count is already the number of incidents at that stop location.
# Sum it per neighborhood.
neighborhood_stats = joined.groupby('AREA_NAME')['Delay_Count'].sum().reset_index()
neighborhood_stats.columns = ['AREA_NAME', 'total_incidents']

# Merge back with neighborhood geometries for mapping
neighborhood_map = neighborhoods_gdf.merge(neighborhood_stats, on='AREA_NAME', how='left')
neighborhood_map['total_incidents'] = neighborhood_map['total_incidents'].fillna(0)  # neighborhoods with no incidents get 0

# -------------------------------
# 5. Create Folium choropleth map
# -------------------------------
# Calculate map center (average of all neighborhood centroids)
center_lat = neighborhoods_gdf.geometry.centroid.y.mean()
center_lon = neighborhoods_gdf.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# Add choropleth layer
folium.Choropleth(
    geo_data=neighborhood_map.to_json(),
    name='Incidents per Neighborhood',
    data=neighborhood_map,
    columns=['AREA_NAME', 'total_incidents'],
    key_on='feature.properties.AREA_NAME',
    fill_color='YlOrRd',          # yellow-orange-red color scale
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name='Total Incidents'
).add_to(m)

# Add tooltips to show neighborhood name and incident count on hover
folium.GeoJsonTooltip(fields=['AREA_NAME', 'total_incidents'],
                      aliases=['Neighborhood:', 'Incidents:'],
                      localize=True).add_to(
    folium.GeoJson(
        neighborhood_map.to_json(),
        style_function=lambda x: {'fillOpacity': 0, 'color': 'black', 'weight': 0.5}
    )
)

# Optionally add a layer control
folium.LayerControl().add_to(m)

# -------------------------------
# 6. Save the map
# -------------------------------
output_file = "assets/data/incidents_by_neighborhood.html"
m.save(output_file)
print(f"✅ Choropleth map saved to {output_file}")

Neighborhood columns: ['AREA_S_CD', 'AREA_NAME', 'geometry']


C:\Users\bains\AppData\Local\Temp\ipykernel_3660\2680564922.py:59: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = neighborhoods_gdf.geometry.centroid.y.mean()
C:\Users\bains\AppData\Local\Temp\ipykernel_3660\2680564922.py:60: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = neighborhoods_gdf.geometry.centroid.x.mean()


✅ Choropleth map saved to assets/data/incidents_by_neighborhood.html


In [20]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# -------------------------------
# 1. Load location analysis data
# -------------------------------
loc_df = pd.read_csv("assets/data/location_analysis.csv")

# Drop rows with missing coordinates
loc_df = loc_df.dropna(subset=['stop_lat', 'stop_lon'])

# Use Delay_Count as the weight for the heatmap (more incidents = hotter)
# Optionally, you could use Total_Delay_Min for severity-weighted heat
heat_data = loc_df[['stop_lat', 'stop_lon', 'Delay_Count']].values.tolist()

# -------------------------------
# 2. Create a base map centered on Toronto
# -------------------------------
center_lat = loc_df['stop_lat'].mean()
center_lon = loc_df['stop_lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# -------------------------------
# 3. Add heatmap layer
# -------------------------------
HeatMap(
    heat_data,
    min_opacity=0.3,
    max_zoom=15,
    radius=15,          # size of each point influence
    blur=10,            # smoothing
    gradient={0.4: 'blue', 0.65: 'lime', 0.8: 'yellow', 1: 'red'}
).add_to(m)

# -------------------------------
# 4. Save the map
# -------------------------------
output_file = "assets/data/hotspot_map.html"
m.save(output_file)
print(f"✅ Hotspot map saved to {output_file}")

✅ Hotspot map saved to assets/data/hotspot_map.html


In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import numpy as np

# -------------------------------
# 1. Load location analysis data
# -------------------------------
loc_df = pd.read_csv("assets/data/location_analysis.csv")

# Drop rows with missing coordinates
loc_df = loc_df.dropna(subset=['stop_lat', 'stop_lon'])

# -------------------------------
# 2. Aggregate per location to get overall average delay
# -------------------------------
# Group by stop coordinates and sum the total delay minutes and delay count
agg = loc_df.groupby(['stop_lat', 'stop_lon']).agg({
    'Total_Delay_Min': 'sum',
    'Delay_Count': 'sum'
}).reset_index()

# Remove any locations that have zero delay count (no incidents) – though they shouldn't exist
agg = agg[agg['Delay_Count'] > 0]

# Compute overall average delay per incident for each location
agg['avg_delay'] = agg['Total_Delay_Min'] / agg['Delay_Count']

# Drop any remaining NaN or infinite values just in case
agg = agg.replace([np.inf, -np.inf], np.nan).dropna(subset=['avg_delay'])

# Prepare data for heatmap: [lat, lon, weight] where weight = avg_delay
heat_data = agg[['stop_lat', 'stop_lon', 'avg_delay']].values.tolist()

# -------------------------------
# 3. Create a base map centered on Toronto
# -------------------------------
center_lat = agg['stop_lat'].mean()
center_lon = agg['stop_lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# -------------------------------
# 4. Add heatmap layer with average delay as weight
# -------------------------------
HeatMap(
    heat_data,
    min_opacity=0.3,
    max_zoom=15,
    radius=15,
    blur=10,
    gradient={0.4: 'blue', 0.65: 'lime', 0.8: 'yellow', 1: 'red'}
).add_to(m)

# -------------------------------
# 5. Save the map
# -------------------------------
output_file = "assets/data/hotspot_avg_delay.html"
m.save(output_file)
print(f"✅ Hotspot map (average delay) saved to {output_file}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1043, in launch_instance
    app.start()
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1043, in launch_instance
    app.start()
  File "c:\Users\bains\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\kernelapp.py", 

AttributeError: _ARRAY_API not found

✅ Hotspot map (average delay) saved to assets/data/hotspot_avg_delay.html


In [1]:
pip install pandas geopandas shapely


[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
#!/usr/bin/env python3
"""
prepare_data.py

Reads location_analysis.csv and the GeoJSON boundary files from assets/data/,
performs spatial joins, and writes aggregated JSON files:
- wards_aggregated.json
- neighbourhoods_aggregated.json
- hotspots_aggregated.json

These files are optimised for client‑side filtering in the TTC Delay Analytics app.
"""

import os
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATA_DIR = "assets/data"          # relative to where the script is run
LOCATION_CSV = os.path.join(DATA_DIR, "location_analysis.csv")
WARD_GEOJSON = os.path.join(DATA_DIR, "gtawards.geojson")
NEIGHBOURHOOD_GEOJSON = os.path.join(DATA_DIR, "toronto.geojson")

OUT_WARDS = os.path.join(DATA_DIR, "wards_aggregated.json")
OUT_NEIGHBOURHOODS = os.path.join(DATA_DIR, "neighbourhoods_aggregated.json")
OUT_HOTSPOTS = os.path.join(DATA_DIR, "hotspots_aggregated.json")

# ----------------------------------------------------------------------
# Load location data
# ----------------------------------------------------------------------
print("📥 Loading location data...")
df = pd.read_csv(LOCATION_CSV)

# Ensure numeric columns are proper numbers
df['stop_lat'] = pd.to_numeric(df['stop_lat'], errors='coerce')
df['stop_lon'] = pd.to_numeric(df['stop_lon'], errors='coerce')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')   # allows NaN if missing
df['Delay_Count'] = pd.to_numeric(df['Delay_Count'], errors='coerce').fillna(0).astype(int)
df['Total_Delay_Min'] = pd.to_numeric(df['Total_Delay_Min'], errors='coerce').fillna(0)

# Drop rows with invalid coordinates
df = df.dropna(subset=['stop_lat', 'stop_lon'])
print(f"   Loaded {len(df)} records with valid coordinates.")

# Create geometry column for spatial joins
geometry = [Point(xy) for xy in zip(df['stop_lon'], df['stop_lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# ----------------------------------------------------------------------
# Wards aggregation
# ----------------------------------------------------------------------
print("\n🏛️ Processing wards...")
wards_gdf = gpd.read_file(WARD_GEOJSON)

# Ensure both are in the same CRS (they are, but just in case)
if wards_gdf.crs != gdf.crs:
    wards_gdf = wards_gdf.to_crs(gdf.crs)

# Spatial join: assign ward name to each point (inner join – keep only points inside a ward)
joined_wards = gpd.sjoin(gdf, wards_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

# Group by ward, year, transit, incident category
wards_agg = joined_wards.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

# Rename columns to match frontend expectations
wards_agg.rename(columns={
    'AREA_NAME': 'ward',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

# Replace NaN years (if any) with null – they will be filtered out later anyway
wards_agg['year'] = wards_agg['year'].where(pd.notna(wards_agg['year']), None)

print(f"   Created {len(wards_agg)} aggregated ward records.")

# Write to JSON
wards_agg.to_json(OUT_WARDS, orient='records', indent=None)   # compact JSON
print(f"   ✅ Saved to {OUT_WARDS}")

# ----------------------------------------------------------------------
# Neighbourhoods aggregation
# ----------------------------------------------------------------------
print("\n🏘️ Processing neighbourhoods...")
neighbourhoods_gdf = gpd.read_file(NEIGHBOURHOOD_GEOJSON)

if neighbourhoods_gdf.crs != gdf.crs:
    neighbourhoods_gdf = neighbourhoods_gdf.to_crs(gdf.crs)

joined_neigh = gpd.sjoin(gdf, neighbourhoods_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

neigh_agg = joined_neigh.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

neigh_agg.rename(columns={
    'AREA_NAME': 'neighbourhood',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

neigh_agg['year'] = neigh_agg['year'].where(pd.notna(neigh_agg['year']), None)

print(f"   Created {len(neigh_agg)} aggregated neighbourhood records.")
neigh_agg.to_json(OUT_NEIGHBOURHOODS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_NEIGHBOURHOODS}")

# ----------------------------------------------------------------------
# Hotspots aggregation (by unique stop)
# ----------------------------------------------------------------------
print("\n🔥 Processing hotspots (stops)...")
# Group by stop coordinates, year, transit, category
hotspot_agg = df.groupby(['stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

hotspot_agg.rename(columns={
    'stop_lat': 'lat',
    'stop_lon': 'lon',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

hotspot_agg['year'] = hotspot_agg['year'].where(pd.notna(hotspot_agg['year']), None)

print(f"   Created {len(hotspot_agg)} aggregated hotspot records.")
hotspot_agg.to_json(OUT_HOTSPOTS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_HOTSPOTS}")

print("\n🎉 All aggregations complete!")

📥 Loading location data...
   Loaded 137143 records with valid coordinates.

🏛️ Processing wards...
   Created 4870 aggregated ward records.
   ✅ Saved to assets/data\wards_aggregated.json

🏘️ Processing neighbourhoods...
   Created 16141 aggregated neighbourhood records.
   ✅ Saved to assets/data\neighbourhoods_aggregated.json

🔥 Processing hotspots (stops)...
   Created 137131 aggregated hotspot records.
   ✅ Saved to assets/data\hotspots_aggregated.json

🎉 All aggregations complete!


## Data Downloading Debug